In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor
from sklearn.linear_model import Ridge


df = pd.read_csv('thailand_house_prices_1000.csv')


df = df.replace(r'^\s*$', np.nan, regex=True)

X = df.drop('Price_THB', axis=1)
y = df['Price_THB']


categorical_features = ['Location', 'Land_Shape', 'Orientation']
numerical_features = ['Area_sqm', 'Bedrooms', 'Bathrooms']


num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])


cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, numerical_features),
        ('cat', cat_transformer, categorical_features)
    ])


ensemble_model = VotingRegressor(
    estimators=[
        ('rf', RandomForestRegressor(n_estimators=100, random_state=42)),
        ('gb', GradientBoostingRegressor(n_estimators=100, random_state=42)),
        ('ridge', Ridge(alpha=1.0))
    ]
)


model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', ensemble_model)
])


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model.fit(X_train, y_train)

print("Model พร้อมใช้งานแล้ว!")

Model พร้อมใช้งานแล้ว!


In [2]:
import joblib

model_data = {
    'model': ensemble_model, 
    'columns': list(X_train.columns)
}
joblib.dump(model_data, 'house_price_rf.pkl') 
print("\n💾 เซฟไฟล์สมอง Ensemble (3 โมเดล) ลงเครื่องสำเร็จ!")


💾 เซฟไฟล์สมอง Ensemble (3 โมเดล) ลงเครื่องสำเร็จ!
